<a href="https://colab.research.google.com/github/lmknijn/jeweled_style/blob/main/Polyptoton_adj_window_sizes_no_repetitions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install git+https://github.com/cwf2/uva_common

In [1]:
# utils
import os
import re
import json
import unicodedata as ud
import string

# for tokenized texts
import uva_common

# for analysis
import pandas as pd
import numpy as np

# for output
from IPython.display import HTML
import seaborn as sns
from matplotlib import pyplot as plt

In [2]:
# select text to analyze
#
# Iliad         tlg0012.tlg001.perseus-grc2
# Odyssey       tlg0012.tlg002.perseus-grc2
# Argonautica   tlg0001.tlg001.perseus-grc2
# Posthomerica  tlg2046.tlg001.perseus-grc2
# Dionysiaca    tlg2045.tlg001.perseus-grc2

urn = "tlg2046.tlg001.perseus-grc2"

In [3]:
# Download tokenized data
#  - for tokenization process see https://github.com/cwf2/uva_common

uva_common.download(f"{urn}.csv", node_id="tokens")

100%|█████████████████████████████████| 11.0M/11.0M [00:01<00:00, 6.48Mbytes/s]


In [4]:
# load downloaded tokens into a Pandas DataFrame
path = os.path.join("data", f"{urn}.csv")
tokens = pd.read_csv(path)
display(tokens)

,author,title,work,urn,line_id,text,lemma,pos,verbform,mood,...,case,gender,speech_id,speaker,addressee,level,type,cluster,turn,tags
0,Quintus Smyrnaeus,τὰ μεθ᾿ Ὅμηρον,Posthomerica,urn:cts:greekLit:tlg2046.tlg001.perseus-grc2:1.1,01_0001,Εὖθʼ,Εὖθʼ,PART,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
1,Quintus Smyrnaeus,τὰ μεθ᾿ Ὅμηρον,Posthomerica,urn:cts:greekLit:tlg2046.tlg001.perseus-grc2:1.1,01_0001,ὑπὸ,ὑπό,ADP,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
2,Quintus Smyrnaeus,τὰ μεθ᾿ Ὅμηρον,Posthomerica,urn:cts:greekLit:tlg2046.tlg001.perseus-grc2:1.1,01_0001,Πηλείωνι,Πηλείων,NOUN,NaN,NaN,...,Dat,Masc,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
3,Quintus Smyrnaeus,τὰ μεθ᾿ Ὅμηρον,Posthomerica,urn:cts:greekLit:tlg2046.tlg001.perseus-grc2:1.1,01_0001,δάμη,δαμάζω,VERB,Fin,Ind,...,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
4,Quintus Smyrnaeus,τὰ μεθ᾿ Ὅμηρον,Posthomerica,urn:cts:greekLit:tlg2046.tlg001.perseus-grc2:1.1,01_0001,θεοείκελος,θεοείκελος,ADJ,NaN,NaN,...,Nom,Masc,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60111,Quintus Smyrnaeus,τὰ μεθ᾿ Ὅμηρον,Posthomerica,urn:cts:greekLit:tlg2046.tlg001.perseus-grc2:1...,14_0658,ὑπὲρ,ὑπέρ,ADP,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
60112,Quintus Smyrnaeus,τὰ μεθ᾿ Ὅμηρον,Posthomerica,urn:cts:greekLit:tlg2046.tlg001.perseus-grc2:1...,14_0658,πόντοιο,πόντος,NOUN,NaN,NaN,...,Gen,Masc,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
60113,Quintus Smyrnaeus,τὰ μεθ᾿ Ὅμηρον,Posthomerica,urn:cts:greekLit:tlg2046.tlg001.perseus-grc2:1...,14_0658,λυγρὰς,λυγρός,ADJ,NaN,NaN,...,Acc,Fem,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
60114,Quintus Smyrnaeus,τὰ μεθ᾿ Ὅμηρον,Posthomerica,urn:cts:greekLit:tlg2046.tlg001.perseus-grc2:1...,14_0658,ὑπάλυξαν,ὑπάλυξαν,VERB,Fin,Ind,...,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN


In [ ]:
# minor changes to make table match Lotte's columns
tokens = tokens.rename(columns={"text": "token"})
tokens[["book", "line"]] = pd.DataFrame(tokens["line_id"].str.split("_").tolist())
tokens["book"] = tokens["book"].str.lstrip("0")
tokens["line"] = tokens["line"].str.lstrip("0")

What needs to happen?

- Divide the tokens into lists of five
- Each token is the first of a new set of five (so 1-5, 2-6, 3-7 etc.)
- Keep each token's id, book and line number (maybe change id into 00_001a and so on?)


In [12]:
# set window size
window_size = 7

# create new list to hold windows
n_tokens = []
# remember last line to increment window id suffix
last_line_id = None

for i in range(len(tokens) - (window_size - 1)): # Ensure there are at least n tokens to form a sequence

    # copy next window_size lines from the table
    this_window = tokens.iloc[i:i+window_size][["book", "line", "token", "lemma", "pos"]]
    
    # assign window id
    line_id = tokens.iloc[i]["line_id"]
    if line_id != last_line_id:
        suffix_counter = 0
    suffix_letter = string.ascii_lowercase[suffix_counter]
    this_window.insert(0, "window_id", line_id + suffix_letter)

    # add this window to growing list
    n_tokens.append(this_window)

# concatenate as a single data frame
n_tokens = pd.concat(n_tokens)

# add combined lemma-pos column
n_tokens["lemma_pos"] = n_tokens[["lemma", "pos"]].apply(tuple, axis=1)

In [13]:
# ### Lotte's original code
# window_size = 7

# all_tokens_data = []
# for index, row in lines.iterrows():
#     for token in row['tokens']:
#         all_tokens_data.append({
#             'token_obj': token,
#             'original_line_id': row['id'],
#             'original_line_book': row['book'],
#             'original_line_line': row['line']
#         })

# # Now create the 5-token sequences
# n_token_sequences = []
# for i in range(len(all_tokens_data) - (window_size - 1)): # Ensure there are at least 5 tokens to form a sequence
#     # Get the 5 token objects
#     sequence_tokens = [all_tokens_data[j]['token_obj'] for j in range(i, i + window_size)]

#     # Get the metadata (id, book, line) from the first token in the sequence
#     first_token_metadata = all_tokens_data[i]

#     n_token_sequences.append({
#         'id': first_token_metadata['original_line_id'],
#         'book': first_token_metadata['original_line_book'],
#         'line': first_token_metadata['original_line_line'],
#         'n_tokens': sequence_tokens
#     })

# # Convert to DataFrame for easier handling and display
# n_token_df = pd.DataFrame(n_token_sequences)

# # Add an alphabetical suffix to the 'id' column
# import string

# new_ids = []
# current_id = None
# suffix_counter = 0

# for index, row in n_token_df.iterrows():
#     if row['id'] != current_id:
#         current_id = row['id']
#         suffix_counter = 0

#     suffix_letter = string.ascii_lowercase[suffix_counter]
#     new_id = f"{row['id']}{suffix_letter}"
#     new_ids.append(new_id)

#     suffix_counter += 1

# n_token_df['id'] = new_ids

# print(f"Created {len(n_token_df)} six-token sequences with unique IDs.")
# display(n_token_df.head())

Next cell: change so that each row is one token from a set of five, e.g. 00_001a

In [14]:
# # start with an empty list of rows
# rows = []

# # iterate over speeches
# for line in n_token_df.itertuples():   # lines

#     # iterate over tokens
#     for token in line.n_tokens:   # tokens

#         # create a record for this token
#         row = {
#             "window_id": line.id,
#             "book": line.book,
#             "line": line.line,
#             "token": token.text,        #token.text
#             "lemma": token.lemma_,
#             "pos": token.pos_,
#             "lemma_pos": (token.lemma_, token.pos_)
#         }

#         # add row to the list of rows
#         rows.append(row)

# # convert to a table
# n_tokens = pd.DataFrame(rows)   # tokens

# display(n_tokens[0:40])  # tokens

In [15]:
from collections import Counter

In [16]:
results = []

pos_mask = n_tokens['pos'].isin(['ADJ', 'NOUN', 'NUM', 'PROPN', 'VERB'])

excluded_lemmas = ["ἀλλ'", "'ἀλλʼ", "ἀλλά", "ἀλλὰ", "ἀτάρ", "ἀτὰρ", "αὐτάρ", "αὐτὰρ", "ἤ", "ἢ", "ἠέ", "ἠὲ", "ἦε", "ἦέ", "ἠδ'", "ἠδʼ" "ἠδέ", "ἠδὲ", "ἤτ'", "ἤτʼ", "ἤτε", "ἤτοι", "μηδ'", "μηδʼ", "μηδέ", "μηδὲ", "μήθ'", "μήτ'", "μήτʼ", "μήτε", "μήτέ", "οὐδ'", "οὐδʼ", "οὐδέ", "οὐδὲ", "οὔθ'", "οὔθʼ", "οὔτ'", "οὔτʼ", "οὔτε", "οὔτέ", "ἀρ'", "ἀρʼ", "ἄρ'", "ἄρʼ", "ἂρ'", "ἂρʼ", "ἄρα", "ἄρά", "ῥ'", "ῥʼ", "ῥα", "ῥά", "αὖ", "γ'", "γʼ", "γε", "γέ", "γάρ", "γὰρ", "δαὶ", "δ'", "δʼ", "δέ", "δὲ", "θην", "θήν", "κε", "κέ", "κεν", "κέν", "χ'", "χʼ", "κ'", "κʼ", "μάν", "μὰν", "μέν", "μὲν", "μήν", "μὴν", "νυ", "νύ", "νυν", "περ", "πέρ", "μὰ", "καί", "καὶ", "μή", "μὴ", "οὐ", "οὐκ", "οὐχ", "οὔ", "οὖν", "ὦ", "ὤ", "ὢ", "που", "πού", "τὰρ", "τε", "τέ", "τ'", "τʼ", "θ'", "θʼ", "ἂν", "ἄν", "δή", "δὴ", "τοιγὰρ", "αἴ", "αἲ", "αἴθ'", "αἴθʼ", "αἴθε", "εἰ", "εἴ", "εἴθ'",  "εἴθʼ", "ἐπεί", "ἐπεὶ", "ἐπειδὰν", "ἐπειδὴ", "ἐπήν", "ἐπὴν", "εὖθ'", "εὖθʼ", "εὖτ'", "εὖτʼ", "εὖτε", "εὖτέ", "ἠύτ'", "ἠύτʼ", "ἠΰτ'", "ἠΰτʼ", "ἠΰτε", "ἦμος", "ἡνίκ", "ἵν'", "ἵνʼ", "ἵνα", "ἵνά", "ἤν", "ἢν", "ὅτ'", "ὅτʼ", "ὅτε", "ὡς", "ὥστ'", "ὥστʼ", "ὥστε", "ὅππως", "ὅπως", "εἷος", "εἷός", "ἕως", "ἧος", "ἧός", "ὄφρ'", "ὄφρʼ", "ὄφρα", "ὄφρά", "πρίν", "πρὶν", "ὁ", "ὅς"]

lexicon_mask = ~n_tokens['lemma'].isin(excluded_lemmas)

combined_mask = pos_mask & lexicon_mask

lemmas_by_window = n_tokens.loc[combined_mask].groupby('window_id').agg(
    book = ('book', 'first'),
    line = ('line', 'first'),
    lemmas = ('lemma', list),
    tokens = ('token', list),
    pos = ('pos', list),
    lemma_pos = ('lemma_pos', list)
    ).reset_index()

display(lemmas_by_window)


,window_id,book,line,lemmas,tokens,pos,lemma_pos
0,01_0001a,1,1,"[Πηλείων, δαμάζω, θεοείκελος, ἕκτωρ, Πηλείων, ...","[Πηλείωνι, δάμη, θεοείκελος, Ἕκτωρ, Πηλείωνι, ...","[NOUN, VERB, ADJ, NOUN, NOUN, VERB, ADJ, NOUN,...","[(Πηλείων, NOUN), (δαμάζω, VERB), (θεοείκελος,..."
1,01_0002a,1,1,"[πυρή, κατέδαψε, ὀστέον, γαῖα, πυρή, κατέδαψε,...","[πυρὴ, κατέδαψε, ὀστέα, γαῖα, πυρὴ, κατέδαψε, ...","[NOUN, VERB, NOUN, NOUN, NOUN, VERB, NOUN, NOU...","[(πυρή, NOUN), (κατέδαψε, VERB), (ὀστέον, NOUN..."
2,01_0003a,1,1,"[τρώς, ἔμιμνον, πρίαμος, πόλις, τρώς, ἔμιμνον,...","[Τρῶες, ἔμιμνον, Πριάμοιο, πόληα, Τρῶες, ἔμιμν...","[NOUN, VERB, NOUN, NOUN, NOUN, VERB, NOUN, NOU...","[(τρώς, NOUN), (ἔμιμνον, VERB), (πρίαμος, NOUN..."
3,01_0004a,1,1,"[δείδω, μένος, ἐύς, θρασύφρων, αἰακίδης, μένο...","[δειδιότες, μένος, ἠῢ, θρασύφρονος, Αἰακίδαο, ...","[VERB, NOUN, ADJ, ADJ, NOUN, NOUN, ADJ, ADJ, N...","[(δείδω, VERB), (μένος, NOUN), ( ἐύς, ADJ), (θ..."
4,01_0005a,1,1,"[ξυλοχοισι, βοῦς, βλοσυρός, λέων, ἔρχομαι, ξυλ...","[ξυλοχοισι, βόες, βλοσυροῖο, λέοντος, ἐλθέμεν,...","[NOUN, NOUN, ADJ, NOUN, VERB, NOUN, NOUN, ADJ,...","[(ξυλοχοισι, NOUN), (βοῦς, NOUN), (βλοσυρός, A..."
...,...,...,...,...,...,...,...
8798,14_0653a,14,14,"[χασσαμένου, πόντος, ἀκτή, ἐριδούπων, πόντος, ...","[χασσαμένου, πόντου, ἀκτάων, ἐριδούπων, πόντου...","[VERB, NOUN, NOUN, ADJ, NOUN, NOUN, ADJ, NOUN,...","[(χασσαμένου, VERB), (πόντος, NOUN), (ἀκτή, NO..."
8799,14_0654a,14,14,"[αἰγιαλός, κατεκτείνω, αἰγιαλός, κατεκτείνω, α...","[αἰγιαλοῖο, κατεκτάθη, αἰγιαλοῖο, κατεκτάθη, α...","[NOUN, VERB, NOUN, VERB, NOUN, VERB, ADJ, VERB...","[(αἰγιαλός, NOUN), (κατεκτείνω, VERB), (αἰγιαλ..."
8800,14_0655a,14,14,"[ἀθάνατος, τελέω, κακός, νόος, τελέω, κακός, ν...","[ἀθανάτων, ἐτέλεσσε, κακὸς, νόος, ἐτέλεσσε, κα...","[ADJ, VERB, ADJ, NOUN, VERB, ADJ, NOUN, NOUN, ...","[(ἀθάνατος, ADJ), (τελέω, VERB), (κακός, ADJ),..."
8801,14_0656a,14,14,"[ἀργεῖος, πλώω, ὅσος, χεῖμα, κέδασσεν, πλώω, ὅ...","[Ἀργεῖοι, πλώεσκον, ὅσους, χεῖμα, κέδασσεν, πλ...","[ADJ, VERB, ADJ, NOUN, VERB, VERB, ADJ, NOUN, ...","[(ἀργεῖος, ADJ), (πλώω, VERB), (ὅσος, ADJ), (χ..."


In [ ]:
# lemmas_by_window is already one row per window_id (see previous cell), so
# there's nothing left to group/sum here — the original re-groupby crashed
# with "TypeError: unsupported operand type(s) for +: 'int' and 'list'"
# because bare `sum` starts its accumulator at 0, which can't be added to a
# single-row group's list value. Kept as a plain copy instead.
results = lemmas_by_window.copy()
results["wc"] = results["lemma_pos"].apply(Counter)
results

In [ ]:
results_limited = results.drop(columns = ['lemmas', 'pos', 'lemma_pos'])

results_limited


In [ ]:
# Identify repeated lemmas (appear at least twice)
results_limited['repeated_lemma'] = results_limited['wc'].apply(                                  # apply a lambda function to the data in wc in results
    lambda c: [lemma for lemma, count in c.items() if count >= 2]                  # a lambda function is an anonymous function, c is the Count object from wc
                                                                                   # If the count in the items (the lemma: count pairs in wc) is two or higher, keep the lemma
)

# Keep couplets where there is at least one such lemma
filtered_results = results_limited[results_limited['repeated_lemma'].apply(len) >= 1].copy()            # from the results, keep only the lines that have one or more from the list of repeated_lemmas

# Create a new DataFrame with one row per repeated lemma
expanded_results = filtered_results.explode('repeated_lemma')

# Extract lemma and pos from the tuple in 'repeated_lemma'
expanded_results['rep_lemma'] = expanded_results['repeated_lemma'].apply(lambda x: x[0])
expanded_results['rep_pos'] = expanded_results['repeated_lemma'].apply(lambda x: x[1])

# Drop the original 'repeated_lemma' column and other intermediate columns
expanded_results = expanded_results.drop(columns=['repeated_lemma'])

# Display the results
display(expanded_results)

Exclude rows with repetitions (two identical tokens instead of two different forms of the same lemma)

In [ ]:
def exclude_repetitions(tokens_list):
    duplicates = set()
    for token in tokens_list:
        if token in duplicates:
            return True
        duplicates.add(token)
    return False

# Filter out rows where the 'tokens' list contains duplicate forms
noreps_expanded_results = expanded_results[~expanded_results['tokens'].apply(exclude_repetitions)].copy()

display(noreps_expanded_results)

In [ ]:
noreps_expanded_results.to_csv('ph_pol_noreps_7win_12_3_26.csv')